In [21]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import AutoModel
from collections import Counter
from sklearn.model_selection import train_test_split
from collections import Counter

In [3]:
CON = Path("/kaggle/input/datasets/zennightshade/intent5/preprocessed/mentalmanip_con.csv")
MAJ = Path("/kaggle/input/datasets/zennightshade/intent5/preprocessed/mentalmanip_maj.csv")

In [17]:


class Vocab:
    def __init__(self, min_freq=2):
        self.min_freq = min_freq
        self.word2idx = {
            "<pad>": 0,
            "<unk>": 1,
            "<p1>": 2,
            "<p2>": 3
        }
        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def normalize(self, text):
        text = text.lower()

        # normalize contractions
        text = re.sub(r"n't", " n't", text)
        text = re.sub(r"'re", " 're", text)
        text = re.sub(r"'s", " 's", text)
        text = re.sub(r"'ll", " 'll", text)

        return text

    def tokenize(self, text):
        text = self.normalize(text)

        # keep punctuation as tokens
        text = re.sub(r"([!?.,])", r" \1 ", text)

        # collapse spaces
        text = re.sub(r"\s+", " ", text).strip()

        return text.split()

    def build(self, texts):
        counter = Counter()

        for text in texts:
            tokens = self.tokenize(text)
            counter.update(tokens)

        for word, freq in counter.items():
            if freq >= self.min_freq and word not in self.word2idx:
                idx = len(self.word2idx)
                self.word2idx[word] = idx
                self.idx2word[idx] = word

    def encode(self, text, speaker=None):
        tokens = self.tokenize(text)

        if speaker == 1:
            tokens = ["<p1>"] + tokens
        elif speaker == 2:
            tokens = ["<p2>"] + tokens

        return [self.word2idx.get(t, 1) for t in tokens]

In [5]:
import torch
from torch.utils.data import Dataset
import re


def parse_dialogue_with_speakers(dialogue):
    pattern = r'(Person\d+):'
    splits = re.split(pattern, dialogue)

    turns = []
    for i in range(1, len(splits), 2):
        speaker = splits[i]
        text = splits[i+1].strip()

        speaker_id = 1 if "1" in speaker else 2
        turns.append((speaker_id, text))

    return turns


def build_transitions(turns):
    pairs = []
    for i in range(1, len(turns)):
        pairs.append((turns[i-1], turns[i]))
    return pairs

class ConversationDataset(Dataset):
    def __init__(self, df, vocab, max_len=50, max_turns=10):
        self.samples = []
        self.vocab = vocab
        self.max_len = max_len
        self.max_turns = max_turns

        for _, row in df.iterrows():
            turns = parse_dialogue_with_speakers(row["Dialogue"])

            if len(turns) < 2:
                continue

            encoded_turns = []

            for s, text in turns[:max_turns]:
                enc = vocab.encode(text, speaker=s)
                enc = enc[:max_len] + [0] * max(0, max_len - len(enc))
                encoded_turns.append(enc)

            self.samples.append((encoded_turns, row["label"]))

    def __getitem__(self, idx):
        turns, label = self.samples[idx]

        return {
            "turns": torch.tensor(turns, dtype=torch.long),  # (T, L)
            "length": len(turns),
            "label": torch.tensor(label, dtype=torch.float)
        }

    def __len__(self):
        return len(self.samples)

In [30]:
def collate_fn(batch):
    batch_size = len(batch)

    max_turns = max(item["turns"].shape[0] for item in batch)
    max_len = batch[0]["turns"].shape[1]

    turns_batch = torch.zeros((batch_size, max_turns, max_len), dtype=torch.long)
    mask = torch.zeros((batch_size, max_turns), dtype=torch.float)

    labels = []

    for i, item in enumerate(batch):
        t = item["turns"]
        length = t.shape[0]

        turns_batch[i, :length] = t
        mask[i, :length] = 1

        labels.append(item["label"])

    return {
        "turns": turns_batch,   # (B, T, L)
        "mask": mask,           # (B, T)
        "label": torch.stack(labels)
    }

In [7]:
import torch
import torch.nn as nn


class ConversationModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_labels=12):
        super().__init__()

        self.hidden_dim = hidden_dim

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.turn_encoder = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.transition_encoder = nn.LSTM(
            hidden_dim * 4,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_labels)
        )

        self.register_buffer("thresholds", torch.full((num_labels,), 0.5))

    def encode_turn(self, x):
        emb = self.embedding(x)

        out, _ = self.turn_encoder(emb)
        h = out.mean(dim=1)

        return h  # (B, 2H)

    def forward(self, batch):
        turns = batch["turns"]   # (B, T, L)
        mask = batch["mask"]     # (B, T)

        B, T, L = turns.shape

        # flatten turns
        turns = turns.reshape(B * T, L)

        h = self.encode_turn(turns)   # (B*T, 2H)
        h = h.reshape(B, T, -1)       # (B, T, 2H)

        # transitions
        diff = h[:, 1:, :] - h[:, :-1, :]
        prod = h[:, 1:, :] * h[:, :-1, :]

        delta = torch.cat([diff, prod], dim=-1)  # (B, T-1, 4H)

        # encode full transition sequence
        out, (h_n, _) = self.transition_encoder(delta)

        # global summary
        global_repr = torch.cat([h_n[-2], h_n[-1]], dim=1)  # (B, 2H)
        
        # local strongest transition
        mask_t = mask[:, 1:].unsqueeze(-1)  # (B, T-1, 1)
        out_masked = out.masked_fill(mask_t == 0, -1e9)
        
        local_repr, _ = torch.max(out_masked, dim=1)  # (B, 2H)
        
        # combine both
        final_repr = torch.cat([global_repr, local_repr], dim=1)  # (B, 4H)
        
        logits = self.classifier(final_repr)

        return logits

    # -------- Inference --------
    def predict_proba(self, batch):
        logits = self.forward(batch)
        return torch.sigmoid(logits)

    def predict(self, batch):
        probs = self.predict_proba(batch)
        return (probs > self.thresholds.to(probs.device)).int()

    def set_thresholds(self, thresholds):
        self.thresholds = torch.tensor(
            thresholds,
            dtype=torch.float,
            device=self.thresholds.device
        )

In [8]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate(model, loader, device, name="Eval"):
    model.eval()

    preds, labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            pred = model.predict(batch).cpu().numpy()
            label = batch["label"].cpu().numpy()

            preds.append(pred)
            labels.append(label)

    preds = np.vstack(preds)
    labels = np.vstack(labels)

    print(f"\n{name} Metrics:")
    print("F1 (macro):", f1_score(labels, preds, average="macro", zero_division=0))
    print("F1 (micro):", f1_score(labels, preds, average="micro", zero_division=0))
    print(f"Thresholds: {model.thresholds}")

In [9]:
from sklearn.metrics import f1_score
import numpy as np

def tune_thresholds(model, loader, device):
    model.eval()
    all_probs, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}

            probs = model.predict_proba(batch).cpu().numpy()
            labels = batch["label"].cpu().numpy()

            all_probs.append(probs)
            all_labels.append(labels)

    all_probs = np.vstack(all_probs)
    all_labels = np.vstack(all_labels)

    thresholds = []

    for i in range(all_labels.shape[1]):
        best_t, best_f1 = 0.5, 0

        for t in np.linspace(0.05, 0.9, 18):
            preds = (all_probs[:, i] > t).astype(int)
            f1 = f1_score(all_labels[:, i], preds, zero_division=0)

            if f1 > best_f1:
                best_f1 = f1
                best_t = t
            # print(f"Class {i} - Best F1: {best_f1}")

        thresholds.append(best_t)

    model.set_thresholds(thresholds)

In [28]:
def train(model, train_loader, val_loader, lr, pos_weight, device, epochs=20):

    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device))

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            logits = model(batch)
            loss = criterion(logits, batch["label"])

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"\nEpoch {epoch+1} Loss: {total_loss / len(train_loader):.4f}")

        # print("yo")
        tune_thresholds(model, val_loader, device)
        # print("yo2")

        evaluate(model, val_loader, device, name=f"Val Epoch {epoch+1}")

In [11]:
def get_stratify_key(label):
    count = sum(label)

    if count == 0:
        return "none"
    elif count == 1:
        return "single"
    else:
        return "multi"   


In [12]:
import pandas as pd
import ast

def load_dataframe(path):
    df = pd.read_csv(path)

    # convert string -> list
    df["label"] = df["label"].apply(lambda x: ast.literal_eval(x))

    return df

df = load_dataframe(CON)

def get_stratify_key(label):
    count = sum(label)

    if count == 0:
        return "none"
    elif count == 1:
        return "single"
    else:
        return "multi"   # 🔥 collapse everything else

def check_distribution(df, name):
    labels = np.vstack(df["label"].values)
    print(f"\n{name}")
    print("Pos:", labels.sum(axis=0))

check_distribution(df, "before split")

df["stratify_key"] = df["label"].apply(get_stratify_key)

from sklearn.model_selection import train_test_split

train_df, val_test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["stratify_key"]
)

val_df, test_df = train_test_split(
    val_test_df,
    test_size=0.5,
    random_state=42,
    stratify=val_test_df["stratify_key"]
)


check_distribution(train_df, "Train")
check_distribution(val_df, "Val")
check_distribution(test_df, "Test")


before split
Pos: [ 87  83  58 213  69  30 384 321 133 361 607 899]

Train
Pos: [ 70  64  51 165  49  23 312 263 108 289 482 720]

Val
Pos: [ 8  9  6 24  9  3 37 27 13 32 59 96]

Test
Pos: [ 9 10  1 24 11  4 35 31 12 40 66 83]


In [18]:
all_turns = []

for dialogue in train_df["Dialogue"]:
    turns = parse_dialogue_with_speakers(dialogue)
    for _, text in turns:
        all_turns.append(text)

vocab = Vocab(min_freq=2)
vocab.build(all_turns)

print("Vocab size:", len(vocab.word2idx))

Vocab size: 5659


In [19]:
lengths = []

for d in df["Dialogue"]:
    turns = parse_dialogue_with_speakers(d)
    for _, text in turns:
        lengths.append(len(vocab.encode(text)))

import numpy as np
print("Mean:", np.mean(lengths))
print("95th percentile:", np.percentile(lengths, 95))
print("Max:", np.max(lengths))

Mean: 16.951578337677383
95th percentile: 49.0
Max: 398


In [31]:
train_dataset = ConversationDataset(train_df, vocab)
val_dataset = ConversationDataset(val_df, vocab)
test_dataset = ConversationDataset(test_df, vocab)


train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn  # THIS is mandatory
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn
)

In [36]:
labels = np.vstack(train_df["label"].values)

pos_counts = labels.sum(axis=0)
neg_counts = len(labels) - pos_counts

pos_weight = torch.tensor((neg_counts / (pos_counts + 1e-5)), dtype=torch.float).to(device)
print(pos_weight)

tensor([158.1356, 181.6826, 257.8340,  40.6906, 274.1773, 868.7063,  13.9150,
         18.7168,  80.2295,  15.9081,   6.2475,   2.7027], device='cuda:0')


In [43]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = ConversationModel(
    vocab_size=len(vocab.word2idx),
    embed_dim= 128,
    hidden_dim= 192,
    num_labels= 12
)

model.to(device)

train(
    model= model, 
    train_loader= train_loader, 
    val_loader= val_loader, 
    lr= 2e-3,
    pos_weight= pos_weight, 
    epochs= 25,
    device= device
)


Epoch 1 Loss: 2.4893

Val Epoch 1 Metrics:
F1 (macro): 0.19572209860330494
F1 (micro): 0.2153267512929008
Thresholds: tensor([0.9000, 0.9000, 0.9000, 0.8000, 0.9000, 0.9000, 0.7000, 0.7500, 0.9000,
        0.7000, 0.7000, 0.5000], device='cuda:0')

Epoch 2 Loss: 2.2295

Val Epoch 2 Metrics:
F1 (macro): 0.20835957306306516
F1 (micro): 0.279324055666004
Thresholds: tensor([0.8500, 0.8500, 0.9000, 0.7500, 0.9000, 0.9000, 0.6500, 0.6000, 0.9000,
        0.7000, 0.6000, 0.0500], device='cuda:0')

Epoch 3 Loss: 2.1460

Val Epoch 3 Metrics:
F1 (macro): 0.2206199660959592
F1 (micro): 0.2808607021517554
Thresholds: tensor([0.8500, 0.9000, 0.8500, 0.5500, 0.9000, 0.9000, 0.5500, 0.6500, 0.7500,
        0.7000, 0.5500, 0.5000], device='cuda:0')

Epoch 4 Loss: 1.9651

Val Epoch 4 Metrics:
F1 (macro): 0.21777118016220268
F1 (micro): 0.2811289429994466
Thresholds: tensor([0.8000, 0.9000, 0.9000, 0.6500, 0.8500, 0.8500, 0.5500, 0.6500, 0.9000,
        0.7500, 0.6000, 0.5000], device='cuda:0')

Epoch

In [44]:
evaluate(model, test_loader, device, name=f"Test time")


Test time Metrics:
F1 (macro): 0.19843482816314859
F1 (micro): 0.3142100617828773
Thresholds: tensor([0.5000, 0.8000, 0.5500, 0.2000, 0.8500, 0.5000, 0.6500, 0.7000, 0.8500,
        0.8000, 0.5500, 0.1000], device='cuda:0')


In [ ]:
import matplotlib.pyplot as plt

models    = ["M1\nMean Pool", "M2\nMax+Mean", "M3\nAttention", "M4\nHierarchical", "M5\nTransitions"]
macro_f1  = [0.2151, 0.2498, 0.2366, 0.1617, 0.2131]
micro_f1  = [0.3360, 0.3567, 0.3492, 0.2844, 0.3522]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

for ax, vals, title, color in [
    (ax1, macro_f1, "Macro F1 — Phase I", "#2c7bb6"),
    (ax2, micro_f1, "Micro F1 — Phase I", "#1a9641"),
]:
    bars = ax.bar(models, vals, color=color, width=0.5, edgecolor="white")
    ax.set_ylim(0, max(vals) * 1.25)
    ax.set_title(title, fontsize=12)
    ax.axhline(max(vals), color="orange", linestyle="--", linewidth=1, alpha=0.7)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.004, f"{v:.3f}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()